In [29]:
import pyspark
from pyspark.sql import SparkSession


spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Taxi_Pipeline") \
    .config("spark.ui.port", "4040") \
    .config("spark.ui.bindAddress", "0.0.0.0") \
    .getOrCreate()


In [30]:
df_green = spark.read.parquet('data/pq/green/*/*')

In [31]:
df_green.registerTempTable('green')

In [41]:
df_green_revenue = spark.sql("""
SELECT 
    date_trunc('hour', lpep_pickup_datetime) AS hour, 
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    green
WHERE
    lpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
""")
df_green_revenue.limit(5).show()

[Stage 55:==================================================>       (7 + 1) / 8]

+-------------------+----+------------------+--------------+
|               hour|zone|            amount|number_records|
+-------------------+----+------------------+--------------+
|2020-01-28 19:00:00| 134|193.61000000000004|            17|
|2020-01-22 19:00:00|  65|            657.03|            41|
|2020-01-27 08:00:00|  17|             85.56|             4|
|2020-01-02 09:00:00|  66|             229.4|            12|
|2020-01-02 12:00:00|  89|310.28000000000003|            14|
+-------------------+----+------------------+--------------+



In [33]:
df_green_revenue \
    .repartition(20) \
    .write.parquet('data/report/revenue/green', mode='overwrite')

26/03/04 12:02:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

In [34]:
df_yellow = spark.read.parquet('data/pq/yellow/*/*')
df_yellow.registerTempTable('yellow')

In [43]:
df_yellow_revenue = spark.sql("""
SELECT 
    date_trunc('hour', tpep_pickup_datetime) AS hour, 
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    yellow
WHERE
    tpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
""")
df_yellow_revenue.limit(5).show()

[Stage 61:===================================================>      (8 + 1) / 9]

+-------------------+----+------------------+--------------+
|               hour|zone|            amount|number_records|
+-------------------+----+------------------+--------------+
|2020-01-30 19:00:00| 163|  9447.85000000002|           530|
|2020-01-16 22:00:00| 100| 4605.539999999999|           250|
|2020-01-03 16:00:00| 132|27982.999999999978|           480|
|2020-01-12 09:00:00| 107|3147.7399999999984|           204|
|2020-01-21 18:00:00| 162| 14090.39000000004|           816|
+-------------------+----+------------------+--------------+



In [44]:
df_yellow_revenue \
    .repartition(20) \
    .write.parquet('data/report/revenue/yellow', mode='overwrite')

26/03/04 12:17:29 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/04 12:17:29 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

In [45]:
df_green_revenue = spark.read.parquet('data/report/revenue/green')
df_yellow_revenue = spark.read.parquet('data/report/revenue/yellow')

In [46]:
df_green_revenue_tmp = df_green_revenue \
    .withColumnRenamed('amount', 'green_amount') \
    .withColumnRenamed('number_records', 'green_number_records')

df_yellow_revenue_tmp = df_yellow_revenue \
    .withColumnRenamed('amount', 'yellow_amount') \
    .withColumnRenamed('number_records', 'yellow_number_records')

In [47]:
df_join = df_green_revenue_tmp.join(df_yellow_revenue_tmp, on=['hour', 'zone'], how='outer')

In [48]:
df_join.printSchema()

root
 |-- hour: timestamp (nullable = true)
 |-- zone: integer (nullable = true)
 |-- green_amount: double (nullable = true)
 |-- green_number_records: long (nullable = true)
 |-- yellow_amount: double (nullable = true)
 |-- yellow_number_records: long (nullable = true)



In [51]:
df_join.limit(10).show()

[Stage 83:========================>                                 (3 + 4) / 7]

+-------------------+----+-----------------+--------------------+------------------+---------------------+
|               hour|zone|     green_amount|green_number_records|     yellow_amount|yellow_number_records|
+-------------------+----+-----------------+--------------------+------------------+---------------------+
|2020-01-01 00:00:00|   3|             NULL|                NULL|              25.0|                    1|
|2020-01-01 00:00:00|   4|             NULL|                NULL|1004.3000000000001|                   57|
|2020-01-01 00:00:00|  40|           168.98|                   8|             89.97|                    5|
|2020-01-01 00:00:00|  45|             NULL|                NULL| 732.4800000000002|                   42|
|2020-01-01 00:00:00|  47|             13.3|                   1|               8.3|                    1|
|2020-01-01 00:00:00|  51|             17.8|                   2|              31.0|                    1|
|2020-01-01 00:00:00|  62|           

In [49]:
df_join.write.parquet('data/report/revenue/total', mode='overwrite')

26/03/04 12:18:00 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

In [52]:
df_join = spark.read.parquet('data/report/revenue/total')

In [56]:
df_zones = spark.read \
    .option("header", "true") \
    .csv('zones/taxi_zone_lookup.csv')

In [63]:
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [57]:
df_result = df_join.join(df_zones, df_join.zone == df_zones.LocationID)

In [58]:
df_result.drop('LocationID', 'zone').write.parquet('tmp/revenue-zones')

26/03/04 12:23:25 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

In [61]:
df_rev_zones = spark.read.parquet('tmp/revenue-zones')

In [62]:
df_rev_zones.limit(5).show()

+-------------------+------------------+--------------------+-------------+---------------------+--------+------------+
|               hour|      green_amount|green_number_records|yellow_amount|yellow_number_records| Borough|service_zone|
+-------------------+------------------+--------------------+-------------+---------------------+--------+------------+
|2020-01-01 00:00:00|              NULL|                NULL|          8.8|                    1|Brooklyn|   Boro Zone|
|2020-01-01 00:00:00|              NULL|                NULL|        34.09|                    1|  Queens|   Boro Zone|
|2020-01-01 00:00:00| 531.0000000000001|                  26|       324.35|                   16|Brooklyn|   Boro Zone|
|2020-01-01 00:00:00|50.900000000000006|                   3|         NULL|                 NULL|   Bronx|   Boro Zone|
|2020-01-01 00:00:00|              23.8|                   1|         NULL|                 NULL|Brooklyn|   Boro Zone|
+-------------------+------------------+